In [ ]:
from lxml import objectify, etree
NS = {
    "nas": "http://www.faa.aero/nas/3.0",
    "base": "http://www.fixm.aero/base/3.0",
    "flight": "http://www.fixm.aero/flight/3.0",
    "foundation": "http://www.fixm.aero/foundation/3.0",
    "xsi": "http://www.w3.org/2001/XMLSchema-instance",
}

def parse_message(doc):
    root = objectify.fromstring(doc.encode("utf-8"))

    # You still typically access by navigation/XPath:
    gufi = root.xpath("string(.//*[local-name()='gufi'])")
    flight = root.xpath(
        ".//*[@xsi:type and contains(@xsi:type, 'NasFlightType')]",
        namespaces=NS
    )[0]

    try:
        print(flight.arrival.get("arrivalPoint"))
        print(flight.flightIdentification.get("aircraftIdentification"))
        print(flight.arrival.runwayPositionAndTime.runwayTime.estimated.get("time"))
    except:
        pass
# list(flight.arrival.runwayPositionAndTime.runwayTime.iterchildren())

def split_xml_documents(text: str):
    docs = []
    current = []

    for line in text.splitlines():
        if line.startswith("<?xml") and current:
            docs.append("\n".join(current))
            current = []
        current.append(line)

    if current:
        docs.append("\n".join(current))

    return docs

with open("../../data/interim/messages.log", "r", encoding="utf-8") as f:
    raw = f.read()

rows = []
docs = split_xml_documents(raw)
for doc in docs:
    parse_message(doc)